# NB-4 — Q-HELDOUT-1: Qwen3-1.7B-Base single-look held-out evaluation

Implements manifest `FMI_manifests_and_decision_rules_v1_2.md` Part B. Evaluation only; no training; existing checkpoints only.

Order enforced by the notebook:
1. NB-3 report must exist and say PASS (B.7.1); every file's hash is re-verified before it is loaded.
2. Development-set reproduction gate (B.7.2): each checkpoint is re-scored on the 925 internal-validation rows and must reproduce the
   archived best-epoch labels with **zero mismatches** (archived logits are bf16-quantized, so logit deviation is recorded and flagged
   above 0.25, not used as a stop). Frozen `off` must equal `native` exactly.
3. Split audit (B.7.3): 3,270 rows, passage-group count, overlap with the development partition recorded.
4. Only after a checkpoint passes its gate is the held-out split scored for it; held-out metrics are neither printed nor archived until
   every gate has passed. A failed gate raises with the deviation report and discards the local held-out files.
5. Analysis per B.5–B.6 (primary shared-passage cluster bootstrap, hierarchical sensitivity, seed descriptives, McNemar) and the
   prespecified wording of B.8, produced mechanically.

Inputs on Drive: `MyDrive/FPDA/…` (checkpoints, HF cache), `MyDrive/FMI/NB3/qwen_hash_precheck.json`, and the handoff archive
`MyDrive/FMI/fpda_model_handoff_2026-09-24.zip` (archived development-set predictions for the gate). A100 recommended
(nine 6.9 GB fp32 checkpoints are loaded in turn; about 1.5–2.5 hours). The runtime is not deleted.

In [ ]:
import time
from datetime import datetime, timezone
RUN_STARTED_PERF = time.perf_counter(); RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
from google.colab import drive
drive.mount('/content/drive'); print('Run started (UTC):', RUN_STARTED_UTC)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'gradio', 'diffusers'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
packages = ['pandas==2.2.3', 'huggingface-hub==0.36.2', 'transformers==4.57.6', 'tokenizers==0.22.1', 'datasets==4.0.0', 'accelerate==1.10.1',
            'sentencepiece>=0.2,<0.3', 'protobuf>=5.29,<7', 'scikit-learn>=1.6,<1.8', 'psutil>=5.9,<8', 'safetensors>=0.4.3,<1']
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--upgrade-strategy', 'only-if-needed', *packages], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
if proc.returncode: print(proc.stdout); raise RuntimeError('Dependency installation failed.')
print('Pinned dependencies installed (same pins as the FPDA wave notebooks).')

In [ ]:
import gc, hashlib, json, math, os, platform, random, shutil, zipfile
from collections import Counter
from importlib.metadata import version as package_version
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset
from safetensors.torch import load_file as load_safetensors
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
try:
    from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU required (A100 recommended).')
DEVICE = torch.device('cuda'); AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16; GPU_NAME = torch.cuda.get_device_name(0)

MANIFEST_ID = 'FMI_manifests_and_decision_rules_v1_2.md / Part B (Q-HELDOUT-1)'
MANIFEST_LOCK_COMMIT = ''           # from MANIFEST_LOCK.md; recorded, not checked
GATE_OVERRIDE_ANNOTATED = True      # 2026-09-25: annotated override, see MANIFEST_LOCK.md (T4-archived wave4b13 cells)
OVERRIDE_LIMITS_DEFAULT = (0.01, 0.25)                 # (max label-mismatch fraction, max abs logit deviation) for A100-archived cells
OVERRIDE_LIMITS_T4 = (0.02, 0.50)                      # widened only for cells whose archive was produced on a Tesla T4 (wave4b13: frozen seed 42)
T4_ARCHIVED_CELLS = {('native_frozen', 42), ('attention_frozen', 42), ('mixer_frozen', 42)}
def override_limits(cell, seed): return OVERRIDE_LIMITS_T4 if (cell, seed) in T4_ARCHIVED_CELLS else OVERRIDE_LIMITS_DEFAULT
BOOTSTRAP_SEED, BOOTSTRAP_REPLICATES = 20260925, 10000
DATA_SEED, MAX_LENGTH, EVAL_BATCH = 2026, 512, 4
MODEL_ID, MODEL_REV = 'Qwen/Qwen3-1.7B-Base', 'ea980cb0a6c2ae4b936e82123acc929f1cec04c1'
DATASET_ID, DATASET_REV = 'google/boolq', '35b264d03638db9f4ce671b711558bf7ff0f80d5'
INVENTORY = [
 {
  "cell": "attention_frozen",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b10",
  "drive_file": "wave4b10_qwen17_corrected_frozen_paths_progress/attention_frozen_seed13_best.safetensors",
  "bytes": 4211252,
  "sha256": "d47383e873f7a830b816fdaba9a5731bb7c4638cbeb843ca635da50413c9558c",
  "regime": "frozen",
  "topology": "attention"
 },
 {
  "cell": "mixer_frozen",
  "seed": 13,
  "best_epoch": 3,
  "wave": "wave4b10",
  "drive_file": "wave4b10_qwen17_corrected_frozen_paths_progress/mixer_frozen_seed13_best.safetensors",
  "bytes": 4211180,
  "sha256": "0b15dba7c9f75f0c200b4b60f8b57012ba0ea1940a1c078d852cc4e49fa434d7",
  "regime": "frozen",
  "topology": "mixer"
 },
 {
  "cell": "attention_frozen",
  "seed": 42,
  "best_epoch": 2,
  "wave": "wave4b13",
  "drive_file": "wave4b13_qwen17_corrected_frozen_seed42_progress/attention_frozen_seed42_best.safetensors",
  "bytes": 4211252,
  "sha256": "3b6516f329d27eadbd993c4f9563be303f5fc853293df87bf5c69fe97c6e27e1",
  "regime": "frozen",
  "topology": "attention"
 },
 {
  "cell": "mixer_frozen",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b13",
  "drive_file": "wave4b13_qwen17_corrected_frozen_seed42_progress/mixer_frozen_seed42_best.safetensors",
  "bytes": 4211180,
  "sha256": "456fc913dfccdb637e00e3f48aa2d78a5e2d10654eb20ac4cb5d2d6a1ce02f11",
  "regime": "frozen",
  "topology": "mixer"
 },
 {
  "cell": "attention_frozen",
  "seed": 71,
  "best_epoch": 2,
  "wave": "wave4b14",
  "drive_file": "wave4b14_qwen17_corrected_frozen_seed71_progress/attention_frozen_seed71_best.safetensors",
  "bytes": 4211252,
  "sha256": "871ce186224563a598645906b08cf3ed9c849cdfb733f4ea353de7abcf676e9b",
  "regime": "frozen",
  "topology": "attention"
 },
 {
  "cell": "mixer_frozen",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b14",
  "drive_file": "wave4b14_qwen17_corrected_frozen_seed71_progress/mixer_frozen_seed71_best.safetensors",
  "bytes": 4211180,
  "sha256": "81d944c261bb5dcffe7cb13dd1f7d5152df87a021a1d483eb63d016f92c8d9f6",
  "regime": "frozen",
  "topology": "mixer"
 },
 {
  "cell": "attention_trainable",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b11",
  "drive_file": "wave4b11_qwen17_corrected_trainable_attention_progress/best_model_path_weights.pt",
  "bytes": 6886644540,
  "sha256": "e1cbb96584eecdbcc4b9efe127bb01a0e1659118158c7ce298aef5d040f42029",
  "regime": "trainable",
  "topology": "attention"
 },
 {
  "cell": "mixer_trainable",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b12",
  "drive_file": "wave4b12_qwen17_corrected_trainable_mixer_progress/best_model_path_weights.pt",
  "bytes": 6886644197,
  "sha256": "ade9bde676492048168522a45dfa5ba1811b5d7485c71f947d404fcac072bee6",
  "regime": "trainable",
  "topology": "mixer"
 },
 {
  "cell": "attention_trainable",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b15",
  "drive_file": "wave4b15_qwen17_corrected_trainable_attention_seed42_progress/best_model_path_weights.pt",
  "bytes": 6886644540,
  "sha256": "462b60feebdea8ec541b4e7a275b1e99a7c108bf90344e1e3cd0cd0819e73ac3",
  "regime": "trainable",
  "topology": "attention"
 },
 {
  "cell": "mixer_trainable",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b16",
  "drive_file": "wave4b16_qwen17_corrected_trainable_mixer_seed42_progress/best_model_path_weights.pt",
  "bytes": 6886644197,
  "sha256": "bcd3792d59db37f126e037353092e3142928d9f7b9f3711663ca1ae084fee613",
  "regime": "trainable",
  "topology": "mixer"
 },
 {
  "cell": "attention_trainable",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b17",
  "drive_file": "wave4b17_qwen17_corrected_trainable_attention_seed71_progress/best_model_path_weights.pt",
  "bytes": 6886644540,
  "sha256": "1b9e597e67dc15f5622c90909944715b338b3b4fcb91c1ddc32c99a6fd7a8c15",
  "regime": "trainable",
  "topology": "attention"
 },
 {
  "cell": "mixer_trainable",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b18",
  "drive_file": "wave4b18_qwen17_corrected_trainable_mixer_seed71_progress/best_model_path_weights.pt",
  "bytes": 6886644197,
  "sha256": "5e8b5317623f510c2dc4c4a80c742a9e12af079a447d76a669264bfc165d1d95",
  "regime": "trainable",
  "topology": "mixer"
 },
 {
  "cell": "no_path",
  "seed": 13,
  "best_epoch": 1,
  "wave": "wave4b8",
  "drive_file": "wave4b8_qwen17_trainable_no_path_progress/best_model_path_weights.pt",
  "bytes": 6882431387,
  "sha256": "dc653687667dffd8be4b3ef6218c51298f2da46b390da22a64bcbb129b1e25c6",
  "regime": "trainable",
  "topology": "none"
 },
 {
  "cell": "no_path",
  "seed": 42,
  "best_epoch": 3,
  "wave": "wave4b19",
  "drive_file": "wave4b19_qwen17_trainable_no_path_seed42_progress/best_model_path_weights.pt",
  "bytes": 6882431387,
  "sha256": "af198e20cd0a2bcb9e99a7d246f84b60b734e360870fa536641e90e9b53ac090",
  "regime": "trainable",
  "topology": "none"
 },
 {
  "cell": "no_path",
  "seed": 71,
  "best_epoch": 3,
  "wave": "wave4b20",
  "drive_file": "wave4b20_qwen17_trainable_no_path_seed71_progress/best_model_path_weights.pt",
  "bytes": 6882431387,
  "sha256": "dfb538ba31aaa3c1e08eee432e95ba0c7f58787ea0ce8424e938fde905dd8b8e",
  "regime": "trainable",
  "topology": "none"
 }
]

DRIVE_FPDA = Path('/content/drive/MyDrive/FPDA'); DRIVE_CACHE = DRIVE_FPDA / 'hf_cache'
DRIVE_FMI = Path('/content/drive/MyDrive/FMI'); DRIVE_OUT = DRIVE_FMI / 'NB4'; DRIVE_OUT.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/fmi_nb4'); RESULT_DIR = WORK / 'results'; HELDOUT_LOCAL = WORK / 'heldout_pending'
for p in (WORK, RESULT_DIR, HELDOUT_LOCAL): p.mkdir(parents=True, exist_ok=True)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(64 * 1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

# Gate B.7.1: NB-3 report must exist and pass
nb3_report = DRIVE_FMI / 'NB3' / 'qwen_hash_precheck.json'
if not nb3_report.is_file(): raise RuntimeError('NB-3 report not found; run NB-3 first (manifest B.7.1).')
nb3 = json.loads(nb3_report.read_text())
if nb3.get('status') != 'PASS': raise RuntimeError('NB-3 reports FAIL; do not proceed.')
print('Gate B.7.1: NB-3 PASS on', nb3.get('utc'))

# Archived development-set predictions come from the handoff archive on Drive
handoff_candidates = list(DRIVE_FMI.glob('fpda_model_handoff_2026-09-24.zip')) + list(DRIVE_FPDA.glob('fpda_model_handoff_2026-09-24.zip'))
if not handoff_candidates: raise FileNotFoundError('Place fpda_model_handoff_2026-09-24.zip under MyDrive/FMI/ (archived dev-set predictions are needed for gate B.7.2).')
HANDOFF_ZIP = handoff_candidates[0]; HANDOFF_SHA = sha256_file(HANDOFF_ZIP)
def handoff_csv(wave, name):
    with zipfile.ZipFile(HANDOFF_ZIP) as zf:
        return pd.read_csv(zf.open(f'fpda_model_handoff_2026-09-24/project/data/derived/{wave}/{name}'))
def handoff_json(wave, name):
    with zipfile.ZipFile(HANDOFF_ZIP) as zf:
        return json.loads(zf.read(f'fpda_model_handoff_2026-09-24/project/data/derived/{wave}/{name}'))
VERSIONS = {n: package_version(n) for n in ['pandas', 'huggingface-hub', 'transformers', 'tokenizers', 'datasets', 'accelerate', 'scikit-learn', 'safetensors']}
print('GPU:', GPU_NAME, '| AMP:', AMP_DTYPE, '| torch', torch.__version__, torch.version.cuda, '| handoff:', HANDOFF_ZIP.name, HANDOFF_SHA[:12])

In [ ]:
# Data: development partition (for the reproduction gate) and the official validation split (held-out), exact serialization.
seed_everything(13)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV, cache_dir=str(DRIVE_CACHE), token=HF_TOKEN, use_fast=True)
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
YES_IDS = tokenizer.encode(' yes', add_special_tokens=False); NO_IDS = tokenizer.encode(' no', add_special_tokens=False)
if len(YES_IDS) != 1 or len(NO_IDS) != 1 or YES_IDS == NO_IDS: raise RuntimeError('verbalizer tokens')
if YES_IDS[0] != 9834 or NO_IDS[0] != 902: raise RuntimeError(f'verbalizer ids changed: {YES_IDS}, {NO_IDS}')
CANDIDATE_TOKEN_IDS = torch.tensor([YES_IDS[0], NO_IDS[0]], dtype=torch.long, device=DEVICE)

train_rows = load_dataset(DATASET_ID, revision=DATASET_REV, split='train', cache_dir=str(DRIVE_CACHE), token=HF_TOKEN)
labels = np.asarray([0 if bool(r['answer']) else 1 for r in train_rows])
groups = np.asarray([hashlib.sha256(str(r['passage']).encode('utf-8')).hexdigest() for r in train_rows])
splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=DATA_SEED)
train_idx, dev_idx = next(splitter.split(np.zeros(len(groups)), labels, groups))
assert len(train_idx) == 8502 and len(dev_idx) == 925 and set(groups[train_idx]).isdisjoint(set(groups[dev_idx]))
DEV_GROUPS = set(groups[train_idx]) | set(groups[dev_idx])

# ---- the locked held-out split is loaded here, once (manifest B.1) ----
heldout_rows = load_dataset(DATASET_ID, revision=DATASET_REV, split='validation', cache_dir=str(DRIVE_CACHE), token=HF_TOKEN)
if len(heldout_rows) != 3270: raise RuntimeError(len(heldout_rows))
H_HASH = [hashlib.sha256(str(r['passage']).encode('utf-8')).hexdigest() for r in heldout_rows]
_cl = {h: i for i, h in enumerate(sorted(set(H_HASH)))}; CLUSTER = np.array([_cl[h] for h in H_HASH]); N_CLUSTERS = len(_cl)
overlap = len(set(H_HASH) & DEV_GROUPS)
split_audit = {'heldout_rows': 3270, 'passage_groups': int(N_CLUSTERS), 'passage_overlap_with_development_partition': int(overlap),
               'gate_B73_zero_overlap': overlap == 0, 'yes_rate': float(np.mean([bool(r['answer']) for r in heldout_rows]))}
print(json.dumps(split_audit, indent=1))
if overlap: print('WARNING: passage overlap with the development partition is a property of the official BoolQ split; recorded for the lock annotations.')

def optional_bos(): return [] if tokenizer.bos_token_id is None else [tokenizer.bos_token_id]
PREFIX_IDS = optional_bos() + tokenizer.encode('Passage:\n', add_special_tokens=False)
def encode_prompt(row):
    passage_ids = tokenizer.encode(str(row['passage']), add_special_tokens=False)
    suffix_ids = tokenizer.encode(f"\nQuestion:\n{row['question']}\nAnswer:", add_special_tokens=False)
    available = MAX_LENGTH - 1 - len(PREFIX_IDS) - len(suffix_ids)
    if available < 1: raise ValueError('Question span exceeds maximum length')
    kept = passage_ids[:available]; return PREFIX_IDS + kept + suffix_ids, len(kept) < len(passage_ids)
class EncodedBoolQ(Dataset):
    def __init__(self, source, indices):
        self.items = []
        for index in indices:
            row = source[int(index)]; prompt, truncated = encode_prompt(row)
            self.items.append({'row_index': int(index), 'input_ids': prompt, 'target': 0 if bool(row['answer']) else 1, 'truncated': bool(truncated)})
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]
def collate(batch):
    padded = tokenizer.pad({'input_ids': [it['input_ids'] for it in batch]}, padding=True, return_tensors='pt')
    return {'input_ids': padded['input_ids'].to(DEVICE), 'attention_mask': padded['attention_mask'].to(DEVICE),
            'targets': torch.tensor([it['target'] for it in batch], dtype=torch.long, device=DEVICE), 'row_indices': [it['row_index'] for it in batch]}
dev_data = EncodedBoolQ(train_rows, dev_idx); heldout_data = EncodedBoolQ(heldout_rows, range(len(heldout_rows)))
DEV_LOADER = DataLoader(dev_data, batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate, num_workers=0)
HELDOUT_LOADER = DataLoader(heldout_data, batch_size=EVAL_BATCH, shuffle=False, collate_fn=collate, num_workers=0)
HELDOUT_TARGET = np.array([it['target'] for it in heldout_data.items]); HELDOUT_POS = HELDOUT_TARGET == 0   # 0 = yes
split_audit['heldout_truncated_rows'] = int(sum(it['truncated'] for it in heldout_data.items))
(RESULT_DIR / 'split_serialization_audit.json').write_text(json.dumps({**split_audit, 'dataset': DATASET_ID, 'revision': DATASET_REV, 'scoring': 'two next-token logits at the last non-pad position', 'yes_id': YES_IDS, 'no_id': NO_IDS}, indent=2))
print('dev rows', len(dev_data), '| held-out rows', len(heldout_data), '| truncated held-out', split_audit['heldout_truncated_rows'])

In [ ]:
# Path modules: exact copies of the corrected identity-preserving forms used by waves 4B-10 … 4B-20.
from typing import Literal
PathState = Literal['on', 'norm_only', 'off']
def _validate(hidden, attention_mask, hidden_size, path_state):
    if hidden.ndim != 3 or hidden.shape[-1] != hidden_size: raise ValueError('Unexpected hidden-state shape')
    if attention_mask.shape != hidden.shape[:2]: raise ValueError('attention_mask shape mismatch')
    if path_state not in {'on', 'norm_only', 'off'}: raise ValueError(f'Unknown path state: {path_state}')
    mask = attention_mask.to(device=hidden.device, dtype=torch.bool)
    if not torch.all(mask.any(dim=1)): raise ValueError('Every sequence must contain a non-padding token')
    return mask
def _logit(p): return math.log(p / (1.0 - p))
class IdentityPreservingAttentionPath(nn.Module):
    def __init__(self, hidden_size, rank, num_heads, dropout=0.1, gate_init=0.1):
        super().__init__()
        if rank % num_heads: raise ValueError('rank must divide num_heads')
        self.hidden_size, self.rank = hidden_size, rank; self.num_heads, self.head_dim = num_heads, rank // num_heads
        self.norm = nn.LayerNorm(hidden_size)
        self.q_proj = nn.Linear(hidden_size, rank, bias=False); self.k_proj = nn.Linear(hidden_size, rank, bias=False)
        self.v_proj = nn.Linear(hidden_size, rank, bias=False); self.out_proj = nn.Linear(rank, hidden_size, bias=False)
        self.dropout = nn.Dropout(dropout); self.gate_logit = nn.Parameter(torch.tensor(_logit(gate_init), dtype=torch.float32))
    @property
    def gate(self): return torch.sigmoid(self.gate_logit)
    def _heads(self, value):
        batch, sequence, _ = value.shape; return value.view(batch, sequence, self.num_heads, self.head_dim).transpose(1, 2)
    def forward(self, hidden, attention_mask, *, path_state='on'):
        mask = _validate(hidden, attention_mask, self.hidden_size, path_state)
        if path_state in {'off', 'norm_only'}: return hidden
        query_mask = mask.unsqueeze(-1); normalized = self.norm(hidden)
        q, k, v = (self._heads(layer(normalized)) for layer in (self.q_proj, self.k_proj, self.v_proj))
        batch, sequence, _ = hidden.shape
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        allowed = mask[:, None, None, :].expand(batch, 1, sequence, sequence)
        causal = torch.ones(sequence, sequence, dtype=torch.bool, device=hidden.device).tril(); allowed = allowed & causal[None, None, :, :]
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
        probabilities = torch.softmax(scores.float(), dim=-1).to(v.dtype)
        mixed = torch.matmul(probabilities, v).transpose(1, 2).contiguous()
        update = self.out_proj(mixed.view(batch, sequence, self.rank)); adapted = hidden + self.gate * self.dropout(update)
        return torch.where(query_mask, adapted, hidden)
class IdentityPreservingMixerPath(nn.Module):
    def __init__(self, hidden_size, rank, kernel_size, dropout=0.1, gate_init=0.1):
        super().__init__()
        self.hidden_size, self.rank, self.kernel_size = hidden_size, rank, kernel_size
        self.norm = nn.LayerNorm(hidden_size)
        self.depthwise = nn.Conv1d(hidden_size, hidden_size, kernel_size, groups=hidden_size, bias=False)
        self.down_proj = nn.Linear(hidden_size, rank, bias=False); self.up_proj = nn.Linear(rank, hidden_size, bias=False)
        self.dropout = nn.Dropout(dropout); self.gate_logit = nn.Parameter(torch.tensor(_logit(gate_init), dtype=torch.float32))
    @property
    def gate(self): return torch.sigmoid(self.gate_logit)
    def forward(self, hidden, attention_mask, *, path_state='on'):
        mask = _validate(hidden, attention_mask, self.hidden_size, path_state)
        if path_state in {'off', 'norm_only'}: return hidden
        numeric_mask = mask.unsqueeze(-1).to(hidden.dtype); normalized = self.norm(hidden) * numeric_mask
        mixed = self.depthwise(F.pad(normalized.transpose(1, 2), (self.kernel_size - 1, 0))).transpose(1, 2) * numeric_mask
        update = self.up_proj(F.gelu(self.down_proj(mixed))); adapted = hidden + self.gate * self.dropout(update)
        return torch.where(mask.unsqueeze(-1), adapted, hidden)
class IdentityPath(nn.Module):
    def forward(self, hidden, attention_mask, *, path_state='native'): return hidden
def make_path(topology):
    if topology == 'attention': return IdentityPreservingAttentionPath(2048, 128, 8, dropout=0.1)
    if topology == 'mixer': return IdentityPreservingMixerPath(2048, 254, 4, dropout=0.1)
    return IdentityPath()
for t in ('attention', 'mixer'):
    if sum(p.numel() for p in make_path(t).parameters()) != 1_052_673: raise RuntimeError('parameter match')
print('Path modules ready (1,052,673 parameters each).')

In [ ]:
# Scoring and evaluation (exactly as in the waves) plus gate helpers.
def last_states(hidden, mask):
    indices = mask.sum(dim=1) - 1; return hidden[torch.arange(hidden.shape[0], device=hidden.device), indices]
def candidate_logits(model, last_hidden):
    weights = model.lm_head.weight.index_select(0, CANDIDATE_TOKEN_IDS)
    bias = None if getattr(model.lm_head, 'bias', None) is None else model.lm_head.bias.index_select(0, CANDIDATE_TOKEN_IDS)
    return F.linear(last_hidden, weights, bias).float()
def metric_dict(targets, predictions):
    t, p = np.asarray(targets), np.asarray(predictions)
    return {'accuracy': float(accuracy_score(t, p)), 'balanced_accuracy': float(balanced_accuracy_score(t, p)), 'macro_f1': float(f1_score(t, p, average='macro')),
            'predicted_yes_rate': float(np.mean(p == 0)), 'true_yes_rate': float(np.mean(t == 0)), 'n': int(len(t))}
@torch.no_grad()
def evaluate(model, path, loader, states):
    model.eval(); path.eval(); out = {}
    for state in states:
        targets, predictions, records, loss_sum = [], [], [], 0.0
        for batch in loader:
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                hidden = model.model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], use_cache=False).last_hidden_state
                adapted = hidden if state == 'native' else path(hidden, batch['attention_mask'], path_state=state)
                logits = candidate_logits(model, last_states(adapted, batch['attention_mask']))
            pred = logits.argmax(dim=1); loss_sum += float(F.cross_entropy(logits, batch['targets'], reduction='sum').cpu())
            gold = batch['targets'].cpu().tolist(); guess = pred.cpu().tolist(); targets.extend(gold); predictions.extend(guess)
            for idx, y, p_, pair in zip(batch['row_indices'], gold, guess, logits.cpu().tolist()):
                records.append({'row_index': idx, 'target': y, 'prediction': p_, 'yes_logit': pair[0], 'no_logit': pair[1], 'state': state})
        m = metric_dict(targets, predictions); m['loss'] = loss_sum / len(targets); out[state] = {'metrics': m, 'records': records}
    return out

def archived_dev(item):
    # archived best-epoch development-set predictions for this checkpoint, by state
    if item['regime'] == 'frozen':
        d = handoff_csv(item['wave'], 'best_internal_validation_predictions.csv'); d = d[d.method == item['cell']]
        return {st: d[d.state == st].set_index('row_index') for st in ('on', 'off')}, int(d.best_epoch.iloc[0])
    d = handoff_csv(item['wave'], 'internal_validation_predictions.csv'); ri = handoff_json(item['wave'], 'run_integrity.json'); be = int(ri['best_epoch'])
    d = d[d.epoch == be]; states = ('native',) if item['topology'] == 'none' else ('on', 'off')
    return {st: d[d.state == st].set_index('row_index') for st in states}, be
def gate_compare(item, dev_eval, archived):
    rows = []
    for st, arch in archived.items():
        mine = pd.DataFrame(dev_eval[st]['records']).set_index('row_index').loc[arch.index]
        mism = int((mine.prediction.to_numpy() != arch.prediction.to_numpy()).sum())
        dev = float(np.max(np.abs(mine[['yes_logit', 'no_logit']].to_numpy() - arch[['yes_logit', 'no_logit']].to_numpy())))
        lim = override_limits(item['cell'], item['seed'])
        rows.append({'cell': item['cell'], 'seed': item['seed'], 'state': st, 'rows': int(len(arch)), 'label_mismatches': mism, 'max_abs_logit_dev': dev, 'logit_drift_flag': dev > 0.25, 'pass': mism == 0,
                     'override_eligible': (mism <= lim[0] * len(arch)) and (dev <= lim[1]), 'override_limits': str(lim)})
    return rows
def row_ok(r): return bool(r['pass'] or (GATE_OVERRIDE_ANNOTATED and r.get('override_eligible', False)))
def native_frozen_archived(wave):
    d = handoff_csv(wave, 'best_internal_validation_predictions.csv'); return d[d.method == 'native_frozen'].set_index('row_index')

In [ ]:
# Frozen block: pristine base model (bf16, as in the frozen waves), native dev gate, six paths (dev gate → held-out).
GATE_ROWS, DEV_METRICS, HELDOUT = [], [], {}   # HELDOUT[(cell, seed, state)] = {'metrics', 'records'}  (kept local until all gates pass)
def verify_and_path(item):
    p = DRIVE_FPDA / item['drive_file']
    if not p.is_file() or p.stat().st_size != item['bytes'] or sha256_file(p) != item['sha256']: raise RuntimeError(f"hash/size mismatch: {item['drive_file']}")
    return p
seed_everything(13)
base_bf16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REV, cache_dir=str(DRIVE_CACHE), token=HF_TOKEN, dtype=AMP_DTYPE, low_cpu_mem_usage=True, attn_implementation='sdpa').to(DEVICE)
base_bf16.config.use_cache = False
for prm in base_bf16.parameters(): prm.requires_grad = False
base_bf16.eval(); identity = IdentityPath().to(DEVICE)
# native (zero-shot) on dev: compare with the three frozen waves' archived native predictions (identical base → identical predictions expected)
dev_native = evaluate(base_bf16, identity, DEV_LOADER, ['native'])
for wave in ('wave4b10', 'wave4b13', 'wave4b14'):
    arch = native_frozen_archived(wave); mine = pd.DataFrame(dev_native['native']['records']).set_index('row_index').loc[arch.index]
    mism = int((mine.prediction.to_numpy() != arch.prediction.to_numpy()).sum()); dev = float(np.max(np.abs(mine[['yes_logit', 'no_logit']].to_numpy() - arch[['yes_logit', 'no_logit']].to_numpy())))
    nseed = {'wave4b10': 13, 'wave4b13': 42, 'wave4b14': 71}[wave]; lim = override_limits('native_frozen', nseed)
    GATE_ROWS.append({'cell': 'native_frozen', 'seed': nseed, 'state': 'native', 'rows': len(arch), 'label_mismatches': mism, 'max_abs_logit_dev': dev, 'logit_drift_flag': dev > 0.25, 'pass': mism == 0,
                      'override_eligible': (mism <= lim[0] * len(arch)) and (dev <= lim[1]), 'override_limits': str(lim)})
    print(f'native vs {wave}: mismatches {mism}, max logit dev {dev:.3f}')
# native held-out is computed unconditionally (it is the frozen off state); it stays withheld until all gates are assessed
HELDOUT[('native_frozen', 0, 'native')] = evaluate(base_bf16, identity, HELDOUT_LOADER, ['native'])['native']
for item in [i for i in INVENTORY if i['regime'] == 'frozen']:
    p = verify_and_path(item); path = make_path(item['topology']).to(DEVICE); path.load_state_dict(load_safetensors(str(p)), strict=True); path.eval()
    dev_eval = evaluate(base_bf16, path, DEV_LOADER, ['on', 'off']); archived, be = archived_dev(item)
    if be != item['best_epoch']: raise RuntimeError(f"best epoch mismatch for {item['cell']} seed {item['seed']}: archive {be} vs inventory {item['best_epoch']}")
    rows = gate_compare(item, dev_eval, archived); GATE_ROWS.extend(rows)
    # frozen off must equal native exactly (B.7.2)
    off_pred = pd.DataFrame(dev_eval['off']['records']).set_index('row_index').sort_index(); nat_pred = pd.DataFrame(dev_native['native']['records']).set_index('row_index').sort_index()
    off_eq = bool((off_pred.prediction.to_numpy() == nat_pred.prediction.to_numpy()).all() and np.allclose(off_pred[['yes_logit', 'no_logit']].to_numpy(), nat_pred[['yes_logit', 'no_logit']].to_numpy()))
    GATE_ROWS.append({'cell': item['cell'], 'seed': item['seed'], 'state': 'off==native', 'rows': len(off_pred), 'label_mismatches': None, 'max_abs_logit_dev': None, 'logit_drift_flag': False, 'pass': off_eq, 'override_eligible': False})
    for st in ('on', 'off'): DEV_METRICS.append({'cell': item['cell'], 'seed': item['seed'], 'state': st, **dev_eval[st]['metrics']})
    ok = all(row_ok(r) for r in rows) and off_eq
    print(f"{item['cell']} seed {item['seed']}: gate {'PASS' if ok else 'FAIL'} ", [(r['state'], r['label_mismatches'], round(r['max_abs_logit_dev'], 3)) for r in rows])
    if ok:
        he = evaluate(base_bf16, path, HELDOUT_LOADER, ['on'])
        HELDOUT[(item['cell'], item['seed'], 'on')] = he['on']; HELDOUT[(item['cell'], item['seed'], 'off')] = HELDOUT[('native_frozen', 0, 'native')]
    del path; gc.collect(); torch.cuda.empty_cache()
del base_bf16; gc.collect(); torch.cuda.empty_cache()
print('Frozen block done. Held-out metrics are withheld until all gates pass.')

In [ ]:
# Trainable block: fp32 base instantiated once; each of the nine checkpoints is hash-verified, loaded, dev-gated, then held-out scored.
seed_everything(13)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REV, cache_dir=str(DRIVE_CACHE), token=HF_TOKEN, dtype=torch.float32, low_cpu_mem_usage=True, attn_implementation='sdpa').to(DEVICE)
model.config.use_cache = False; model.eval()
for prm in model.parameters(): prm.requires_grad = False
CKPT_META = []
for item in [i for i in INVENTORY if i['regime'] == 'trainable']:
    t0 = time.time(); p = verify_and_path(item)
    payload = torch.load(p, map_location='cpu', weights_only=False)   # own hash-verified artifact (payload holds plain tensors and metadata)
    if int(payload.get('epoch', -1)) != item['best_epoch']: raise RuntimeError(f"checkpoint epoch {payload.get('epoch')} != inventory {item['best_epoch']} for {item['cell']} seed {item['seed']}")
    CKPT_META.append({'cell': item['cell'], 'seed': item['seed'], 'protocol_id': payload.get('protocol_id'), 'epoch': payload.get('epoch'), 'score': payload.get('score')})
    model.load_state_dict(payload['model_state'], strict=True)
    path = make_path(item['topology']).to(DEVICE)
    if item['topology'] != 'none': path.load_state_dict(payload['path_state'], strict=True)
    path.eval(); del payload; gc.collect()
    states = ['native'] if item['topology'] == 'none' else ['on', 'off']
    dev_eval = evaluate(model, path, DEV_LOADER, states); archived, be = archived_dev(item)
    rows = gate_compare(item, dev_eval, archived); GATE_ROWS.extend(rows)
    for st in states: DEV_METRICS.append({'cell': item['cell'], 'seed': item['seed'], 'state': st, **dev_eval[st]['metrics']})
    ok = all(row_ok(r) for r in rows)
    print(f"{item['cell']} seed {item['seed']}: gate {'PASS' if ok else 'FAIL'} ", [(r['state'], r['label_mismatches'], round(r['max_abs_logit_dev'], 3)) for r in rows], f'({(time.time()-t0)/60:.1f} min)')
    if ok:
        he = evaluate(model, path, HELDOUT_LOADER, states)
        for st in states: HELDOUT[(item['cell'], item['seed'], st)] = he[st]
    del path; gc.collect(); torch.cuda.empty_cache()
del model; gc.collect(); torch.cuda.empty_cache()
GATES = pd.DataFrame(GATE_ROWS); GATES.to_csv(RESULT_DIR / 'gate_B72_dev_reproduction.csv', index=False); pd.DataFrame(DEV_METRICS).to_csv(RESULT_DIR / 'dev_reproduction_metrics.csv', index=False)
pd.DataFrame(CKPT_META).to_csv(RESULT_DIR / 'trainable_checkpoint_meta.csv', index=False)
STRICT_PASS = bool(GATES['pass'].all()); ALL_GATES_PASS = bool(all(row_ok(r) for r in GATE_ROWS)); OVERRIDE_ROWS = [r for r in GATE_ROWS if not r['pass'] and row_ok(r)]
print('\nGATE B.7.2:', 'PASS (strict)' if STRICT_PASS else ('PASS WITH ANNOTATED OVERRIDE on %d rows' % len(OVERRIDE_ROWS) if ALL_GATES_PASS else 'FAIL'), '| logit-drift flags:', int(GATES['logit_drift_flag'].sum()))
display(GATES)
if not ALL_GATES_PASS:
    shutil.rmtree(HELDOUT_LOCAL, ignore_errors=True); HELDOUT.clear()
    raise RuntimeError('Development-set reproduction gate FAILED beyond the annotated override limits (default ≤1% / ≤0.25; T4-archived cells ≤2% / ≤0.5); held-out results discarded. Investigate and annotate MANIFEST_LOCK.md before any re-run.')

In [ ]:
# Analysis per B.5–B.6; the held-out split has now been scored once for every checkpoint.
def ba_of(key): return 100 * HELDOUT[key]['metrics']['balanced_accuracy']
def pred_of(key): return pd.DataFrame(HELDOUT[key]['records']).sort_values('row_index').prediction.to_numpy()
SEEDS = (13, 42, 71); TOPO = {'attention': ('attention_frozen', 'attention_trainable'), 'mixer': ('mixer_frozen', 'mixer_trainable')}
run_rows, pred_rows = [], []
for key, v in HELDOUT.items():
    run_rows.append({'cell': key[0], 'seed': key[1], 'state': key[2], **v['metrics']})
    for r in v['records']: pred_rows.append({'cell': key[0], 'seed': key[1], **r, 'passage_cluster': int(CLUSTER[r['row_index']])})
RUN = pd.DataFrame(run_rows).drop_duplicates(['cell', 'seed', 'state']).sort_values(['cell', 'seed', 'state']); PRED = pd.DataFrame(pred_rows).drop_duplicates(['cell', 'seed', 'state', 'row_index'])
RUN.to_csv(RESULT_DIR / 'heldout_run_results.csv', index=False); PRED.to_csv(RESULT_DIR / 'heldout_predictions.csv', index=False)

# point estimates (seed-level)
def cell_ba(cell, seed, st): return ba_of((cell, seed, st))
est = {}
for T, (fr, tr) in TOPO.items():
    rows = []
    for s in SEEDS:
        dpf = cell_ba(fr, s, 'on') - cell_ba(fr, s, 'off'); dpt = cell_ba(tr, s, 'on') - cell_ba(tr, s, 'off')
        rows.append({'topology': T, 'seed': s, 'frozen_on': cell_ba(fr, s, 'on'), 'frozen_off': cell_ba(fr, s, 'off'), 'delta_path_frozen': dpf, 'trainable_on': cell_ba(tr, s, 'on'), 'trainable_off': cell_ba(tr, s, 'off'),
                     'delta_path_trainable': dpt, 'delta_regime': dpf - dpt, 'no_path': cell_ba('no_path', s, 'native'), 'enabled_minus_no_path': cell_ba(tr, s, 'on') - cell_ba('no_path', s, 'native'),
                     'frozen_on_minus_zero_shot': cell_ba(fr, s, 'on') - ba_of(('native_frozen', 0, 'native')), 'no_path_minus_zero_shot': cell_ba('no_path', s, 'native') - ba_of(('native_frozen', 0, 'native'))})
    est[T] = pd.DataFrame(rows)
SEEDTAB = pd.concat(est.values()); SEEDTAB.to_csv(RESULT_DIR / 'heldout_estimands_by_seed.csv', index=False)
att_minus_mixer = pd.DataFrame([{'seed': s, 'regime': rg, 'attention_minus_mixer': cell_ba(TOPO['attention'][i], s, 'on') - cell_ba(TOPO['mixer'][i], s, 'on')} for s in SEEDS for i, rg in enumerate(('frozen', 'trainable'))])

# cluster-level sufficient statistics for every (cell, seed, state)
cols, cidx = [], {}
def add(key, vec): cidx[key] = len(cols); cols.append(np.bincount(CLUSTER, weights=vec.astype(float), minlength=N_CLUSTERS))
add('npos', HELDOUT_POS.astype(float)); add('nneg', (~HELDOUT_POS).astype(float))
for key in HELDOUT:
    c = pred_of(key) == HELDOUT_TARGET; add(key + ('cp',), (c & HELDOUT_POS).astype(float)); add(key + ('cn',), (c & ~HELDOUT_POS).astype(float))
M = np.stack(cols, axis=1)
def ba_from(S, key): return 50 * (S[cidx[key + ('cp',)]] / S[cidx['npos']] + S[cidx[key + ('cn',)]] / S[cidx['nneg']])
def quantities(S, seeds):
    q = {}
    for T, (fr, tr) in TOPO.items():
        dr, dpf, dpt, enp = [], [], [], []
        for s in seeds:
            a = ba_from(S, (fr, s, 'on')) - ba_from(S, (fr, s, 'off')); b = ba_from(S, (tr, s, 'on')) - ba_from(S, (tr, s, 'off'))
            dpf.append(a); dpt.append(b); dr.append(a - b); enp.append(ba_from(S, (tr, s, 'on')) - ba_from(S, ('no_path', s, 'native')))
        q[f'delta_regime_{T}'] = float(np.mean(dr)); q[f'delta_path_frozen_{T}'] = float(np.mean(dpf)); q[f'delta_path_trainable_{T}'] = float(np.mean(dpt)); q[f'enabled_minus_no_path_{T}'] = float(np.mean(enp))
    for i, rg in enumerate(('frozen', 'trainable')):
        q[f'attention_minus_mixer_{rg}'] = float(np.mean([ba_from(S, (TOPO['attention'][i], s, 'on')) - ba_from(S, (TOPO['mixer'][i], s, 'on')) for s in seeds]))
    return q
rng = np.random.default_rng(BOOTSTRAP_SEED); primary, hier = [], []
for _ in range(BOOTSTRAP_REPLICATES):
    idx = rng.integers(0, N_CLUSTERS, N_CLUSTERS); primary.append(quantities(M[idx].sum(axis=0), SEEDS))           # shared-passage: one resample for all seeds/conditions
for _ in range(BOOTSTRAP_REPLICATES):
    seeds_r = [SEEDS[i] for i in rng.integers(0, 3, 3)]                                                              # hierarchical: seeds, then independent clusters per seed
    S_by_seed = {s: M[rng.integers(0, N_CLUSTERS, N_CLUSTERS)].sum(axis=0) for s in set(seeds_r)}
    acc = {}
    for s in seeds_r:
        qs = quantities(S_by_seed[s], [s])
        for k, v in qs.items(): acc.setdefault(k, []).append(v)
    hier.append({k: float(np.mean(v)) for k, v in acc.items()})
def ci(samples, k): v = np.array([x[k] for x in samples]); return [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
point = {k: float(np.mean([quantities(M.sum(axis=0), [s])[k] for s in SEEDS])) for k in primary[0]}
BOOT = {k: {'point': point[k], 'primary_shared_passage_ci95': ci(primary, k), 'hierarchical_ci95': ci(hier, k)} for k in primary[0]}
# seed-level descriptives and McNemar
desc = {}
for T in TOPO:
    d = est[T]
    for q in ('delta_regime', 'delta_path_frozen', 'delta_path_trainable', 'enabled_minus_no_path'):
        v = d[q].to_numpy(); desc[f'{q}_{T}'] = {'mean': float(v.mean()), 'sd': float(v.std(ddof=1)), 'min': float(v.min()), 'max': float(v.max()), 'values': [float(x) for x in v], 'positive_seeds': int((v > 0).sum())}
def exact_mcnemar_p(x, y):
    n = x + y
    if n == 0: return 1.0
    k = min(x, y); return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
mcn = []
for T, (fr, tr) in TOPO.items():
    for s in SEEDS:
        for cell in (fr, tr):
            cx, cy = pred_of((cell, s, 'on')) == HELDOUT_TARGET, pred_of((cell, s, 'off')) == HELDOUT_TARGET; a, b = int((cx & ~cy).sum()), int((~cx & cy).sum())
            mcn.append({'cell': cell, 'seed': s, 'comparison': 'on_vs_off', 'on_only_correct': a, 'off_only_correct': b, 'p_exact': exact_mcnemar_p(a, b)})
MCN = pd.DataFrame(mcn); MCN.to_csv(RESULT_DIR / 'heldout_mcnemar.csv', index=False)

# B.8 wording (mechanical)
def wording(T):
    lo, hi = BOOT[f'delta_regime_{T}']['primary_shared_passage_ci95']; P = BOOT[f'delta_regime_{T}']['point']; pos = desc[f'delta_regime_{T}']['positive_seeds']
    if lo > 0: w = f'the frozen-minus-trainable dependence difference is positive for the {T} path on the held-out set (+{P:.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}]); direction replicates the encoder result'
    elif hi < 0: w = f'the difference is negative for the {T} path ({P:+.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}])'
    elif P > 0: w = f'the point estimate for the {T} path is positive but the interval includes zero; inconclusive at this sample ({P:+.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}])'
    else: w = f'the point estimate for the {T} path is zero or negative and the interval includes zero; inconclusive at this sample ({P:+.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}])'
    return w + f'; seed-level direction consistency {pos}/3.'
report = {'manifest': MANIFEST_ID, 'zero_shot_native_BA': ba_of(('native_frozen', 0, 'native')), 'seed_level': desc, 'bootstrap': BOOT, 'attention_minus_mixer_by_seed': att_minus_mixer.to_dict('records'),
          'prespecified_wording': {T: wording(T) for T in TOPO}, 'note': 'Intervals including zero are inconclusive; equivalence is not claimed; the headroom explanation is a hypothesis (B.9).'}
(RESULT_DIR / 'heldout_analysis_summary.json').write_text(json.dumps(report, indent=2))
print('Zero-shot base BA: %.2f' % report['zero_shot_native_BA']); display(SEEDTAB.round(2))
for T in TOPO: print(f"\n{T}: Δ_regime {BOOT[f'delta_regime_{T}']['point']:+.2f} primary CI {np.round(BOOT[f'delta_regime_{T}']['primary_shared_passage_ci95'], 2).tolist()} hierarchical {np.round(BOOT[f'delta_regime_{T}']['hierarchical_ci95'], 2).tolist()} | enabled−no-path {BOOT[f'enabled_minus_no_path_{T}']['point']:+.2f} CI {np.round(BOOT[f'enabled_minus_no_path_{T}']['primary_shared_passage_ci95'], 2).tolist()}")
for T in TOPO: print('\nB.8 wording:', report['prespecified_wording'][T])

In [ ]:
# Environment, manifest, archive, Drive copy, download.
env = {'notebook': 'NB-4 Q-HELDOUT-1', 'manifest': MANIFEST_ID, 'manifest_lock_commit': MANIFEST_LOCK_COMMIT, 'gate_override_annotated': GATE_OVERRIDE_ANNOTATED, 'gate_strict_pass': STRICT_PASS,
       'override_rows': [{k: r[k] for k in ('cell', 'seed', 'state', 'label_mismatches', 'max_abs_logit_dev', 'override_limits')} for r in OVERRIDE_ROWS], 'override_limits_default': OVERRIDE_LIMITS_DEFAULT, 'override_limits_t4': OVERRIDE_LIMITS_T4, 'nb3_report_utc': nb3.get('utc'),
       'handoff_zip_sha256': HANDOFF_SHA, 'gpu': GPU_NAME, 'amp_dtype': str(AMP_DTYPE), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'packages': VERSIONS, 'python': platform.python_version(),
       'heldout_evaluated': True, 'single_look': True, 'bootstrap_seed': BOOTSTRAP_SEED, 'bootstrap_replicates': BOOTSTRAP_REPLICATES, 'passage_clusters': int(N_CLUSTERS),
       'run_started_utc': RUN_STARTED_UTC, 'run_finished_utc': datetime.now(timezone.utc).isoformat(), 'active_seconds': time.perf_counter() - RUN_STARTED_PERF}
(RESULT_DIR / 'run_environment.json').write_text(json.dumps(env, indent=2))
rows = [{'file': p.name, 'bytes': p.stat().st_size, 'sha256': sha256_file(p)} for p in sorted(RESULT_DIR.iterdir()) if p.is_file() and p.name != 'FILE_MANIFEST_SHA256.csv']
pd.DataFrame(rows).to_csv(RESULT_DIR / 'FILE_MANIFEST_SHA256.csv', index=False)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); zip_path = Path(shutil.make_archive(f'/content/fmi_nb4_qwen_heldout_results_{stamp}', 'zip', RESULT_DIR))
with zipfile.ZipFile(zip_path) as zf:
    if zf.testzip(): raise RuntimeError('ZIP corrupt')
shutil.copy2(zip_path, DRIVE_OUT / zip_path.name); print('Result ZIP:', zip_path, '| sha256', sha256_file(zip_path), '| Drive:', DRIVE_OUT / zip_path.name)
from google.colab import files
files.download(str(zip_path))